# Trassenfinder speed sweep

Re-queries a subset of the collected segments at several **booked maximum
speeds** (`v_max` in the `zugcharakteristik`), holding route, station pair and
train constant. It exists to estimate one coefficient the main collection
cannot: how energy responds to speed.

## Why the main sample cannot answer this

`01` fires one request per segment per composition with `v_max` taken from the
composition. Both values in use, 200 and 230 km/h, sit above the line speed on
essentially every path, so the booked speed never binds:

- the 230 km/h compositions are **0.08 min slower** on average than the 200 km/h
  ones on the same segment, median difference exactly 0
- between-segment SD of average speed is **11.94 km/h**; within-segment SD
  across all eight compositions is **1.10 km/h**

Those figures come from the pre-2026-08-30 collection, whose braking parameters
capped every train near 110 km/h. The conclusion survives the correction — `01`
still sends one booked speed per composition, so speed is still never varied —
but the numbers will change when `01` is re-run.

So about 99% of the variation in average speed is a property of the line, not
of the train, and the small remainder is driven by mass (median correlation
between mass and speed inside a segment is **-0.92** — the heavy train
accelerates more slowly). Fitting a speed term on that data returns a negative
coefficient: it is reading "slow route" as "slow train", and slow routes are
branch lines and station approaches, which genuinely cost more per tonne-km.

## The experiment

Fix the segment, fix the composition, vary only `v_max`. The sweep must go
**downward** from 200 km/h — upward is a no-op because line speed binds. What
varies within a group is then the booked speed alone, so a within-group
estimator identifies the aerodynamic term without the route confound.

## Cost

`len(SEGMENTS) x len(SWEEP_COMPOSITIONS) x len(SPEED_GRID)` requests, plus a
short pre-flight. The defaults below are 24 x 3 x 6 = **432 requests**, roughly
12-15 minutes. That is about a quarter of what `01` costs.

Runs top to bottom. Writes `data/samples_speed.csv` and
`data/failures_speed.csv`. Does not touch the main sample.

## 1. Setup

Everything that talks to the API lives in `trassenfinder.py`, shared with `01`,
so the sweep uses the same payload template, the same `mutter` resolution and
the same error handling. The only thing this notebook changes is `v_max`.

In [ ]:
import numpy as np
import pandas as pd

import trassenfinder as tf
from data_sources import DATA_DIR, ensure_local, source_input

DATA_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- configuration

# Booked maximum speeds to sweep, km/h. 200 anchors the sweep to the main
# sample: at v_max=200 and schnellfahrstrecken_meiden=True a 200 km/h
# composition reproduces its samples_all row, which is the check in section 6.
#
# The 2026-08-30 runs found the ceiling twice, and the second time it was not
# the line. With bremshundertstel at 70 the train could not exceed about 110
# km/h whatever was booked, so every grid point above 120 returned identical
# results and allowing high-speed lines moved realised speed by 3.2 km/h. That
# was a payload defect, now fixed: braking is derived per composition and lands
# near 195-201 BrH, which measured 157 km/h on Hamburg-Hannover.
#
# So the grid spans the full range again. Where the real ceiling now sits is an
# open question rather than a known quantity: section 7 reports the speed gained
# per grid point, and if the top of the grid still returns duplicates, move
# those points down rather than assuming the cap is physical.
#
# 230 is included for the two new-build compositions and CLAMPED PER
# COMPOSITION: a 200 km/h train is never booked above its own maximum, which
# would be a request for a run it cannot make. The 2026-08-30 sweep topped out
# at 130 km/h realised even on high-speed track, so whether 230 buys anything
# over 200 is a question this run answers rather than one already settled.
SPEED_GRID = [60, 90, 120, 150, 180, 200, 230]

# A factorial design over mass and length, not a mass range.
#
#   composition    mass t   length m   t/m    v_max
#   REF-COUCH-6     313.0     158.4    1.98    200
#   NEW-BAL-7       313.8     185.7    1.69    230   <- same mass, +17% length
#   REF-BAL-9       501.4     237.6    2.11    200   <- mid-range anchor
#   REF-BUD-12      636.0     316.8    2.01    200
#   NEW-BAL-14      627.6     371.4    1.69    230   <- same mass, +17% length
#
# The earlier sweep used three refurbished compositions, all 200 km/h and all
# inside a 1.98-2.11 t/m band, so mass and length were collinear and only their
# product was identifiable. That matters because the model form ties drag to
# MASS, and the physics does not: aerodynamic drag depends on frontal area and
# on skin friction along the train's length. Mass belongs in rolling resistance
# and acceleration.
#
# The 2026-08-30 collection shows the two apart. Holding mass constant and
# adding 17% of length costs 3.4% energy on the twelve-coach pair and 5.2% on
# the six-coach pair - and NEW-BAL-14 is LIGHTER than REF-BUD-12 while using
# more. A coefficient fitted only on refurbished stock and applied to new-build
# stock therefore carries a systematic bias, on exactly the refurbished-versus-
# new comparison the tool exists to make.
#
# The two mass-matched pairs are what separate the terms. Do not drop one half
# of a pair to save requests: the pairing is the design.
SWEEP_COMPOSITIONS = [
    "REF-COUCH-6",
    "NEW-BAL-7",
    "REF-BAL-9",
    "REF-BUD-12",
    "NEW-BAL-14",
]

# How many segments to sweep is set per distance band in section 2 (PER_BAND),
# not as a single total: the bands are not equally informative and an even split
# spends most of the run where the speed cap barely binds.
RANDOM_SEED = 42

# ------------------------------------------------------------------ strata
#
# schnellfahrstrecken_meiden changes which LINE the train runs on, not how fast
# it runs on one line. It is therefore a stratum, held constant within a group,
# never a swept variable — toggling it inside a group would confound the speed
# response with a route change, which is the exact confound this notebook
# exists to break. The group key in 02 is (segment, composition, sfs_allowed).
#
# Why bother with the SFS stratum at all, given night trains do not use
# high-speed lines (night-time maintenance windows)? Because we are identifying
# the TRAIN's drag coefficient, not its route. Drag does not care which line it
# is on, and the steeper NBS gradients are constant within a group so the fixed
# effect absorbs them. What it buys is a second, independent estimate of k over
# the 120-200 range that the conventional network cannot reach.
#
# Carry one caveat: NBS are tunnel-heavy and tunnel drag is materially higher
# than open-air drag. If the SFS stratum returns a visibly larger k, suspect
# that before concluding the v^2 form is wrong.
STRATA = {
    "conventional": True,  # schnellfahrstrecken_meiden
    "sfs": False,
}

# Corridors chosen BECAUSE they are high-speed, not sampled by distance band.
#
# The 2026-08-30 sweep topped out at 130 km/h realised and booked 230 gained
# nothing, which looked like a network ceiling. It was not. It was a property of
# 21 segments picked by distance band: only 5 of them rerouted onto high-speed
# line at all, and those were partial NBS runs. Reading "absent from this
# sample" as "does not occur" is the same error as the bremshundertstel one, a
# limit of the data taken for a property of the world.
#
# The German network has 250-300 km/h line, and the target network is European,
# where LGVs run 300-320. A 230 km/h composition WILL reach its own maximum on
# those, so the calibration has to cover the range or the v^2 term extrapolates
# where drag dominates.
#
# Station pairs, not routes: Trassenfinder picks the path, and with
# schnellfahrstrecken_meiden False on these corridors it will take the fast one.
# Codes are resolved in the pre-flight and anything invalid is dropped, so an
# entry that turns out wrong costs one request, not a run.
NBS_CORRIDORS = [
    ("KK", "FF", "Koeln - Frankfurt (300 km/h)"),
    ("HH", "BLS", "Hannover - Berlin (250)"),
    ("NN", "MH", "Nuernberg - Muenchen via Ingolstadt (300)"),
    ("HH", "FF", "Hannover - Frankfurt via Wuerzburg (280)"),
    ("NN", "UE  P", "Nuernberg - Erfurt (300)"),
    ("FF", "MH", "Frankfurt - Muenchen"),
    ("AH", "FF", "Hamburg - Frankfurt"),
    ("KK", "MH", "Koeln - Muenchen"),
    # Insurance. The pre-flight drops any DS100 Trassenfinder rejects, and a
    # dropped corridor cannot be recovered without another full run, so the list
    # is longer than the minimum it needs to be.
    ("RM", "TS", "Mannheim - Stuttgart (280)"),
    ("BLS", "MH", "Berlin - Muenchen via Erfurt (300)"),
    ("HH", "MH", "Hannover - Muenchen"),
]

# Only segments at least this long get the SFS stratum. Short segments rarely
# have a high-speed parallel, so the request returns the same route and the same
# numbers — duplicated rows, no new information. Section 6 reports how many
# segments actually rerouted, so this threshold can be tuned from evidence.
SFS_MIN_DISTANCE_KM = 200

# Route weighting. None keeps the template's 40/30/30, identical to the main
# collection. See section 6: because energy carries 30% of the weighting, a
# lower v_max can make the optimiser prefer a different path, which breaks the
# "same route" premise. Setting this to a distance-only weighting pins the route
# harder but makes the sweep no longer comparable with samples_all, so it is a
# fallback, not the default.
GEWICHTUNG = None
# GEWICHTUNG = {"streckenlaenge_prozent": 100, "fahrzeit_prozent": 0, "energie_prozent": 0}

# The sweep fires several requests per segment back to back, so space them.
tf.MIN_INTERVAL_S = 0.4

compositions = pd.read_csv(source_input("compositions.csv"))
sweep_compositions = compositions[
    compositions["composition_id"].isin(SWEEP_COMPOSITIONS)
].reset_index(drop=True)

missing = set(SWEEP_COMPOSITIONS) - set(sweep_compositions["composition_id"])
if missing:
    raise ValueError(f"unknown composition_id: {sorted(missing)}")

print("Compositions in the sweep:")
print(
    sweep_compositions[
        ["composition_id", "coaches_gross_weight_80pct_t_wagenzugmasse", "v_max_kmh"]
    ].to_string(index=False)
)


def grid_for(composition):
    """Booked speeds valid for one composition — never above its own v_max."""
    return [v for v in SPEED_GRID if v <= float(composition["v_max_kmh"])]


print("\nSpeed grid:", SPEED_GRID)
for _, comp in sweep_compositions.iterrows():
    print(
        f"  {comp['composition_id']:14s} v_max {comp['v_max_kmh']:.0f} -> "
        f"{grid_for(comp)}"
    )
print("Strata (schnellfahrstrecken_meiden):", STRATA)

## 2. Choosing the segments

Drawn from `samples_all.csv` rather than from the route lists, so every segment
in the sweep is one that already collected successfully for all three
compositions. A segment that failed in `01` will fail here too, and burning six
requests to rediscover that wastes a quarter of the run.

Stratified across distance bands. Short segments are where the speed response
should be weakest (the train spends most of the leg accelerating and braking
rather than cruising) and long ones where it should be strongest, so a sample
concentrated at either end would misstate the coefficient.

In [ ]:
samples = pd.read_csv(ensure_local("samples_all.csv"))
samples["segment_id"] = (
    samples["route_name"].astype(str)
    + "__"
    + samples["start_ds100"].astype(str)
    + "__"
    + samples["end_ds100"].astype(str)
)

# One row per segment, keeping the identifying columns and the distance the
# main collection measured (used only for stratification).
segments = (
    samples[samples["composition_id"].isin(SWEEP_COMPOSITIONS)]
    .groupby("segment_id")
    .agg(
        source=("source", "first"),
        route_name=("route_name", "first"),
        start_stop_name=("start_stop_name", "first"),
        start_ds100=("start_ds100", "first"),
        end_stop_name=("end_stop_name", "first"),
        end_ds100=("end_ds100", "first"),
        n_comps=("composition_id", "nunique"),
        distance_km=("distance_km", "median"),
    )
    .reset_index()
)

# Only segments that collected cleanly for every composition in the sweep.
complete = segments[segments["n_comps"] == len(SWEEP_COMPOSITIONS)]
print(
    f"{len(complete)} of {len(segments)} segments complete for all sweep compositions"
)

BANDS = [0, 50, 100, 200, 400, 2000]
complete = complete.assign(band=pd.cut(complete["distance_km"], BANDS))

# Segments per band, in BANDS order. Stated rather than derived from a total:
# integer division of a total across bands silently discards the remainder, so
# the knob would not mean what it says.
#
# Weighted towards the long end on purpose. On a short leg the train spends most
# of its length accelerating and braking and never reaches 200 km/h anyway, so
# lowering the booked cap changes little: those groups contribute almost no
# variation in realised speed, and what the cap does to an acceleration profile
# is not a v^2 effect. The long bands are where the cap binds across most of the
# run and where k is actually estimable. The short bands are kept as a check on
# whether the fitted k misbehaves down there, not to identify it.
# Trimmed from [3, 3, 4, 5, 6] once the high-speed corridors were added: the
# corridors cover the fast end far better than another long sampled segment
# would, and the sampled draw is there for range and for the mass/length
# contrast, not for speed reach.
PER_BAND = [2, 2, 3, 4, 4]
assert len(PER_BAND) == len(BANDS) - 1, "PER_BAND must have one entry per band"

# Built with an explicit loop rather than groupby().apply(): since pandas 2.2
# apply() drops the grouping column from the result, which would silently take
# "band" out of the frame the next cell reports on.
selected = pd.concat(
    [
        g.sample(min(len(g), n), random_state=RANDOM_SEED)
        for n, (_, g) in zip(PER_BAND, complete.groupby("band", observed=True))
    ]
).reset_index(drop=True)

short = [
    f"{b}: wanted {n}, have {len(g)}"
    for n, (b, g) in zip(PER_BAND, complete.groupby("band", observed=True))
    if len(g) < n
]
if short:
    print("Bands short of the requested allocation:")
    for line in short:
        print("  ", line)

print(f"\nSelected {len(selected)} segments")
print(
    selected.groupby("band", observed=True)
    .agg(
        n=("segment_id", "size"),
        d_min=("distance_km", "min"),
        d_median=("distance_km", "median"),
        d_max=("distance_km", "max"),
    )
    .round(1)
    .to_string()
)

selected["segment_set"] = "sampled"

# The high-speed corridors are appended rather than sampled: they exist to cover
# a speed range the distance-stratified draw cannot reach, so leaving their
# presence to chance would defeat the point.
nbs = pd.DataFrame(
    [
        {
            "segment_id": f"NBS__{start}__{end}",
            "source": "nbs",
            "route_name": name,
            "start_stop_name": start,
            "start_ds100": start,
            "end_stop_name": end,
            "end_ds100": end,
            "n_comps": len(SWEEP_COMPOSITIONS),
            "distance_km": float("nan"),
            "band": pd.NA,
            "segment_set": "nbs",
        }
        for start, end, name in NBS_CORRIDORS
    ]
)
selected = pd.concat([selected, nbs], ignore_index=True)
print(f"\nPlus {len(nbs)} high-speed corridors -> {len(selected)} segments total")

# Distance is unknown for the corridors until they are queried, and they are
# there precisely to be run with high-speed line available.
selected["sfs_stratum"] = (selected["distance_km"] >= SFS_MIN_DISTANCE_KM) | (
    selected["segment_set"] == "nbs"
)

points_per_composition = sum(
    len(grid_for(comp)) for _, comp in sweep_compositions.iterrows()
)
n_conventional = len(selected) * points_per_composition
n_sfs = int(selected["sfs_stratum"].sum()) * points_per_composition
n_requests = n_conventional + n_sfs

print(
    f"\nSegments getting the SFS stratum (>= {SFS_MIN_DISTANCE_KM} km): "
    f"{int(selected['sfs_stratum'].sum())} of {len(selected)}"
)
print(
    f"Collection requests: {n_conventional} conventional + {n_sfs} sfs = {n_requests}"
)
print(
    f"Estimated runtime:   {n_requests * (tf.MIN_INTERVAL_S + 1.3) / 60:.0f}-"
    f"{n_requests * (tf.MIN_INTERVAL_S + 2.2) / 60:.0f} min"
)

## 3. Station pre-flight

Same reasoning as in `01`: `mutter` is a property of the Betriebsstelle, so it
is resolved once per station before any collection starts. Only the stations in
the selected segments are probed, and the cache is shared with the main
collection through the module.

Every station here already resolved during `01`, so this should report all
valid. If it does not, the infrastructure version has moved since that run,
which matters for section 6.

In [ ]:
stations = sorted(set(selected["start_ds100"]) | set(selected["end_ds100"]))
probe_composition = sweep_compositions.iloc[0]

invalid = tf.resolve_stations(stations, probe_composition)

print("Stations resolved:", len(tf.MUTTER_CACHE), "/", len(stations))

if invalid:
    print("\nINVALID:", invalid)
    keep = ~(
        selected["start_ds100"].isin(invalid) | selected["end_ds100"].isin(invalid)
    )
    print(f"Dropping {(~keep).sum()} of {len(selected)} segments")
    selected = selected[keep].reset_index(drop=True)
    print(
        "\nThese resolved during 01 but not now, so Trassenfinder's "
        "infrastructure version has changed. Re-read section 6 before pooling "
        "this sweep with samples_all."
    )
else:
    print("\nAll stations valid")

## 4. Collect

Segment x composition x speed. Failures are logged and skipped so one
unroutable combination cannot stop the run.

Expect some failures at the low end of the grid. A booked speed well under line
speed can make a path unavailable under `knotenbahnhoefe_meiden`, and slow
trains are refused on some sections outright. Those are informative, not fatal:
they tell you where the operating envelope actually ends.

In [ ]:
results = []
failures = []

total = n_requests
done = 0

for _, segment in selected.iterrows():
    strata = (
        STRATA if segment["sfs_stratum"] else {"conventional": STRATA["conventional"]}
    )
    for stratum, avoid_sfs in strata.items():
        for _, composition in sweep_compositions.iterrows():
            for v_max in grid_for(composition):
                done += 1
                try:
                    result = tf.query(
                        segment["start_ds100"],
                        segment["end_ds100"],
                        composition,
                        v_max_kmh=v_max,
                        gewichtung=GEWICHTUNG,
                        vermeidung={"schnellfahrstrecken_meiden": avoid_sfs},
                    )

                    results.append(
                        {
                            "source": "speed",
                            "segment_set": segment["segment_set"],
                            "stratum": stratum,
                            "sfs_allowed": not avoid_sfs,
                            "route_name": segment["route_name"],
                            "start_stop_name": segment["start_stop_name"],
                            "start_ds100": segment["start_ds100"],
                            "end_stop_name": segment["end_stop_name"],
                            "end_ds100": segment["end_ds100"],
                            "composition_id": composition["composition_id"],
                            "n_coaches": composition["n_coaches"],
                            "weight_t": composition[
                                "coaches_gross_weight_80pct_t_wagenzugmasse"
                            ],
                            "length_m": composition["coaches_length_m_wagenzuglaenge"],
                            "v_max_kmh": composition["v_max_kmh"],
                            "bremshundertstel": tf.bremshundertstel(composition),
                            "streckenklasse": tf.streckenklasse(composition),
                            "v_max_requested_kmh": v_max,
                            "energy_kwh": result["energy_kwh"],
                            "energy_components_kwh": result["energy_components_kwh"],
                            "energy_traktion_kwh": result["energy_traktion_kwh"],
                            "energy_hilfsbetriebe_kwh": result[
                                "energy_hilfsbetriebe_kwh"
                            ],
                            "energy_wagen_kwh": result["energy_wagen_kwh"],
                            "distance_km": result["distance_km"],
                            "travel_time_min": result["travel_time_min"],
                            "trassenpreis_eur": result["trassenpreis_eur"],
                            "stationspreis_eur": result["stationspreis_eur"],
                            "preis_energie_eur": result["preis_energie_eur"],
                            "kosten_fahrzeuge_personal_eur": result[
                                "kosten_fahrzeuge_personal_eur"
                            ],
                            "marktsegment": result["marktsegment"],
                            "v_peak_kmh": result["v_peak_kmh"],
                            "v_mean_dist_kmh": result["v_mean_dist_kmh"],
                            "v_rms_kmh": result["v_rms_kmh"],
                            "schnellfahrt_share_pct": result["schnellfahrt_share_pct"],
                            "n_route_points": result["n_route_points"],
                            "maximalwerte": result["maximalwerte"],
                            "speed_unzulaessig": result["speed_unzulaessig"],
                        }
                    )

                except (tf.TrassenfinderError, ValueError) as e:
                    failures.append(
                        {
                            "source": "speed",
                            "segment_set": segment["segment_set"],
                            "stratum": stratum,
                            "sfs_allowed": not avoid_sfs,
                            "route_name": segment["route_name"],
                            "start_ds100": segment["start_ds100"],
                            "end_ds100": segment["end_ds100"],
                            "composition_id": composition["composition_id"],
                            "v_max_requested_kmh": v_max,
                            "error": str(e),
                        }
                    )

    print(f"  {done}/{total} | ok: {len(results)} | failed: {len(failures)}")

print(f"\nCollected {len(results)}, failed {len(failures)}")

if failures:
    fail_df = pd.DataFrame(failures)
    print("\nFailures by requested speed and stratum:")
    print(
        fail_df.pivot_table(
            index="v_max_requested_kmh",
            columns="stratum",
            values="error",
            aggfunc="size",
        ).to_string()
    )
    print("\nFirst few:")
    for f in failures[:5]:
        print(
            f"  {f['start_ds100']} -> {f['end_ds100']} @ "
            f"{f['v_max_requested_kmh']} | {f['error'][:90]}"
        )

## 5. Save

In [ ]:
speed_df = pd.DataFrame(results)
speed_df.to_csv(DATA_DIR / "samples_speed.csv", index=False)
pd.DataFrame(failures).to_csv(DATA_DIR / "failures_speed.csv", index=False)

print(f"{len(speed_df)} samples -> samples_speed.csv")
print(f"{len(failures)} failures -> failures_speed.csv")
print(
    "\nUpload data/samples_speed.csv to the Drive folder "
    "(ENERGY_DRIVE_FOLDER_ID) so 02 can run on a machine that has not "
    "collected."
)

## 6. Did the route actually stay fixed?

The premise of the experiment is that only speed changed. That premise is not
guaranteed: `gewichtung_parameter` puts 30% of the route choice weight on
energy, so a lower `v_max` can make a different path optimal. When the returned
`distance_km` moves across the grid, that group is no longer a controlled
speed experiment and cannot be used to estimate the coefficient.

Two things to read here:

1. **Distance stability per group**, where a group is now
   `(segment, composition, stratum)`. Groups whose distance varies by more than
   a rounding step are flagged. `02` uses only the stable ones. Note that the
   two strata are *expected* to differ from each other — that is the point —
   so stability is only ever checked within a stratum, never across.
2. **The 200 km/h anchor.** For a 200 km/h composition, the `v_max=200` row
   should reproduce its `samples_all` row. A large gap means the infrastructure
   version has moved since August and the two samples cannot be pooled.

If too few groups survive, re-run with the distance-only `GEWICHTUNG` in
section 1. That pins the route but makes the sweep incomparable with
`samples_all`, so the speed coefficient would then have to be estimated
purely within the sweep.

In [ ]:
speed_df["segment_id"] = (
    speed_df["route_name"].astype(str)
    + "__"
    + speed_df["start_ds100"].astype(str)
    + "__"
    + speed_df["end_ds100"].astype(str)
)
speed_df["avg_speed_kmh"] = speed_df["distance_km"] / (speed_df["travel_time_min"] / 60)

GROUP_KEYS = ["segment_id", "composition_id", "stratum"]

stability = speed_df.groupby(GROUP_KEYS).agg(
    n=("energy_kwh", "size"),
    d_min=("distance_km", "min"),
    d_max=("distance_km", "max"),
)
stability["d_spread_pct"] = (
    (stability["d_max"] - stability["d_min"]) / stability["d_min"] * 100
)
stability["route_stable"] = stability["d_spread_pct"] < 0.5

print(f"Groups (segment x composition x stratum): {len(stability)}")
print(f"Route stable across the grid:   {stability['route_stable'].sum()}")
print(f"Route changed:                  {(~stability['route_stable']).sum()}")
print("\nDistance spread within a group, percent:")
print(stability["d_spread_pct"].describe().round(2).to_string())

if (~stability["route_stable"]).any():
    print("\nGroups where the optimiser picked a different path:")
    print(
        stability[~stability["route_stable"]]
        .sort_values("d_spread_pct", ascending=False)
        .head(10)
        .round(2)
        .to_string()
    )

# --- the 200 km/h anchor -----------------------------------------------------
# samples_all was collected with high-speed lines avoided, so only the
# conventional stratum is comparable.
anchor = speed_df[
    (speed_df["v_max_requested_kmh"] == 200)
    & (speed_df["v_max_kmh"] == 200)
    & (speed_df["stratum"] == "conventional")
].merge(
    samples[["segment_id", "composition_id", "energy_kwh", "distance_km"]],
    on=["segment_id", "composition_id"],
    how="inner",
    suffixes=("_sweep", "_main"),
)

if len(anchor):
    anchor["energy_diff_pct"] = (
        (anchor["energy_kwh_sweep"] - anchor["energy_kwh_main"])
        / anchor["energy_kwh_main"]
        * 100
    )
    print(f"\n200 km/h anchor: {len(anchor)} rows matched against samples_all")
    print(anchor["energy_diff_pct"].describe().round(3).to_string())
    if anchor["energy_diff_pct"].abs().median() > 1.0:
        print(
            "\nWARNING: the anchor rows do not reproduce samples_all. The "
            "infrastructure version has moved. Do not pool; re-run 01 or "
            "estimate everything inside this sweep."
        )
else:
    print("\nNo anchor rows — no 200 km/h composition in SWEEP_COMPOSITIONS.")

## 7. First look at the speed response

Before any fitting, look at the shape. Two things to check.

**Sign.** Within a group, energy should now *rise* with booked speed. If it
does not, either the route is not fixed (section 6) or `v_max` is still not
binding on these paths, and the sweep needs a lower grid.

**Curvature.** Aerodynamic drag scales with the square of speed, so energy per
km should curve upward, not rise linearly.

Watch for a turning point at the bottom of the grid. Auxiliary consumption is
counted (`energieverbrauch_hilfsbetriebe_und_wagen_beachten`), so a very slow
run spends longer drawing hotel and auxiliary power. If energy stops falling
below some speed, that is a real effect with an operating implication, not an
artefact, and it means a plain `v^2` term will misfit the bottom of the range.

In [ ]:
import matplotlib.pyplot as plt

stable_keys = set(stability[stability["route_stable"]].index)
stable = speed_df[speed_df.set_index(GROUP_KEYS).index.isin(stable_keys)].copy()

print(f"{len(stable)} rows in route-stable groups\n")

print("Median kWh/km by requested speed and composition")
print(
    stable.assign(kwh_km=stable["energy_kwh"] / stable["distance_km"])
    .pivot_table(
        index="v_max_requested_kmh",
        columns=["stratum", "composition_id"],
        values="kwh_km",
        aggfunc="median",
    )
    .round(2)
    .to_string()
)

# Binding PER GRID POINT, per stratum. This is the diagnostic that matters and
# the one an earlier version of this notebook got wrong: a span measured across
# the whole grid looked healthy (19-27 km/h per distance band) while three of
# six grid points were in fact exact duplicates, because all the movement
# happened in the bottom third. Read the per-point gaps, not the total span.
by_point = stable.pivot_table(
    index="v_max_requested_kmh",
    columns="stratum",
    values="avg_speed_kmh",
    aggfunc="median",
)
print("\nMedian realised avg speed by booked speed and stratum:")
print(by_point.round(1).to_string())

if "nbs" in set(stable["segment_set"]):
    print("\nMedian realised avg speed by booked speed and segment set:")
    print(
        stable.pivot_table(
            index="v_max_requested_kmh",
            columns=["segment_set", "stratum"],
            values="avg_speed_kmh",
            aggfunc="median",
        )
        .round(1)
        .to_string()
    )
    reach = stable[stable["segment_set"] == "nbs"]["avg_speed_kmh"].max()
    print(f"\nHighest realised average on a high-speed corridor: {reach:.1f} km/h")
    print(
        "  This is the range the drag term is identified over. Anything the "
        "backend routes above it is extrapolation, and drag is quadratic, so "
        "the error grows fast."
    )

if "v_rms_kmh" in stable.columns:
    prof = stable[stable["v_rms_kmh"] > 0].copy()
    prof["convexity_pct"] = ((prof["v_rms_kmh"] / prof["avg_speed_kmh"]) ** 2 - 1) * 100
    print("\nConvexity gap: how much a v^2 law on RMS speed exceeds one on the")
    print("average, as a percentage of the drag term.")
    print(
        prof.groupby("segment_set")["convexity_pct"]
        .describe()[["mean", "50%", "max"]]
        .round(1)
        .to_string()
    )
    print(
        "\n  The deployed model uses average speed, because that is all "
        "CountryLeg carries. This is the size of the error that choice costs, "
        "measured rather than assumed - and it is why the profile is collected."
    )
    print("\nHigh-speed line as a share of route distance, by segment set:")
    print(
        stable.groupby(["segment_set", "stratum"])["schnellfahrt_share_pct"]
        .median()
        .round(1)
        .to_string()
    )

print("\nSpeed gained by each grid point over the one below:")
print(by_point.diff().round(2).to_string())
print(
    "\nA gap near zero means that point returned the same run as the one "
    "below and its requests bought nothing. Drop it, or move it into the range "
    "that still binds."
)

print("\nDistinct realised speeds per group (more is better; 2 is the floor):")
print(
    stable.groupby(GROUP_KEYS)["avg_speed_kmh"]
    .nunique()
    .value_counts()
    .sort_index()
    .to_string()
)

# --- the contrast the factorial design exists to resolve -------------------
# Within a mass-matched pair the trains weigh the same and differ by 17% in
# length. If drag scaled with mass, the two would sit on top of each other at
# every speed. If they separate, and separate MORE as speed rises, the gap is
# aerodynamic and the model needs a length term rather than a mass one.
PAIRS = [("REF-COUCH-6", "NEW-BAL-7"), ("REF-BUD-12", "NEW-BAL-14")]

for ref_id, new_id in PAIRS:
    pair = stable[stable["composition_id"].isin([ref_id, new_id])]
    if pair["composition_id"].nunique() < 2:
        continue
    wide = pair.pivot_table(
        index="v_max_requested_kmh",
        columns="composition_id",
        values="energy_kwh",
        aggfunc="median",
    ).dropna()
    if wide.empty or ref_id not in wide or new_id not in wide:
        continue
    wide["gap_pct"] = (wide[new_id] / wide[ref_id] - 1) * 100
    print(f"\n{new_id} vs {ref_id} — same mass, +17% length")
    print(wide.round(2).to_string())
    print(
        "  A gap that widens with booked speed is aerodynamic and belongs in "
        "a length term. A flat gap is a rolling-resistance offset and belongs "
        "in the per-km constant."
    )

# Did allowing high-speed lines actually change the route? Where it did not,
# the SFS rows duplicate the conventional ones and buy nothing; where it did,
# they are a genuinely independent group.
if "sfs" in set(stable["stratum"]):
    widths = stable.pivot_table(
        index=["segment_id", "composition_id"],
        columns="stratum",
        values="distance_km",
        aggfunc="median",
    ).dropna()
    widths["rerouted"] = ~np.isclose(widths["conventional"], widths["sfs"], rtol=0.005)
    print(
        f"\nSFS stratum rerouted in {int(widths['rerouted'].sum())} of "
        f"{len(widths)} segment x composition pairs"
    )
    print(
        "Pairs where it did not reroute duplicate the conventional rows and "
        "can be cut from the next run via SFS_MIN_DISTANCE_KM."
    )
    if widths["rerouted"].any():
        print("\nDistance with and without high-speed lines, where it changed:")
        print(widths[widths["rerouted"]].round(1).to_string())

# Energy within each group, indexed to its own 200 km/h value, so groups of very
# different length can be read on one axis.
# Indexed to each group's own top grid point, not to a fixed 230: the 200 km/h
# compositions never reach it, and indexing them to a missing row would drop
# them from the plot entirely.
top = stable.groupby(GROUP_KEYS)["v_max_requested_kmh"].transform("max")
ref = stable[stable["v_max_requested_kmh"] == top].set_index(GROUP_KEYS)["energy_kwh"]
stable["energy_index"] = stable.apply(
    lambda r: r["energy_kwh"] / ref.get(tuple(r[k] for k in GROUP_KEYS), np.nan),
    axis=1,
)

fig, ax = plt.subplots(figsize=(7, 4.5))
for _, g in stable.groupby(GROUP_KEYS):
    g = g.sort_values("v_max_requested_kmh")
    ax.plot(g["v_max_requested_kmh"], g["energy_index"], alpha=0.25, color="grey")

profile = stable.groupby("v_max_requested_kmh")["energy_index"].median()
ax.plot(
    profile.index,
    profile.values,
    marker="o",
    linewidth=2,
    color="black",
    label="median",
)
ax.axhline(1.0, linewidth=0.8, color="black", linestyle=":")
ax.set_xlabel("booked v_max [km/h]")
ax.set_ylabel("energy, indexed to each group's top grid point")
ax.set_title("Energy response to booked speed, route held fixed")
ax.legend()
plt.tight_layout()
plt.show()

print("\nMedian energy relative to the top of the grid:")
print(profile.round(3).to_string())

## 8. Handover

### Generated files

| File | Contents |
|---|---|
| `calib/data/samples_speed.csv` | One row per segment x composition x booked speed |
| `calib/data/failures_speed.csv` | Combinations Trassenfinder refused, with the message and the speed |

### Columns

Same as `samples_*.csv`, plus three:

- **`v_max_requested_kmh`** — the swept booked speed. `v_max_kmh` still carries
  the composition's nominal maximum.
- **`stratum`** — `conventional` or `sfs`.
- **`sfs_allowed`** — whether high-speed lines were permitted.

`02` groups on `(segment_id, composition_id, stratum)`, so the two strata are
never mixed inside a group.

### What `02` does with this

Two-stage. The speed coefficient is estimated here, from within-group variation
in route-stable groups only, and then held fixed while the level coefficients
are refitted on `samples_all`. Estimating everything jointly on the pooled data
would let the between-route confound back in through the main sample, which is
the whole thing this sweep exists to avoid.

### What the composition set is for

Five compositions in two mass-matched pairs plus a mid-range anchor. Within a
pair the mass is held and the length differs by 17%, which is the only way to
tell a drag term in mass apart from one in length — between-route data cannot,
because across the refurbished fleet mass and length move together at a
near-constant 2 t/m.

Both new-build compositions are 230 km/h, so the sweep also covers the speed
range the refurbished stock cannot reach.

### Limitations

- **Germany only**, like every other source here. This sweep says nothing about
  terrain, and nothing about French LGVs at 300-320 km/h, which the target
  network reaches and Trassenfinder cannot query.
- **Average speed, not a speed profile.** Drag is convex in speed, so a leg
  averaging 130 km/h with a 230 km/h high-speed section and slow approaches
  burns more than one held steady at 130. The model fits and predicts on the
  average, which is consistent only while the profile shape is similar between
  calibration and application. High-speed corridors are in the sweep partly to
  keep that assumption honest.
- **Booked speed, not driven speed.** `v_max` caps the run; realised average
  speed still depends on the line. Section 7 reports how much of the grid
  actually binds.
- **Selection.** The segments are drawn from those that collected successfully
  in `01`, which are already biased towards main-line pairs.
- **One weighting.** The route optimiser stays at 40/30/30. Groups where that
  produced a different path at a different speed are excluded rather than
  corrected.